In [0]:
import requests
import json
import os
from datetime import datetime

In [0]:
BASE_PATH = "dbfs:/Volumes/workspace/project_data_football_raw/pontuacao_raw"

os.makedirs(BASE_PATH, exist_ok=True)

In [0]:
def run():
    print("Iniciando ingestão de pontuação...")

    status = requests.get("https://api.cartola.globo.com/mercado/status").json()
    rodada_atual = status["rodada_atual"]

#loop pra buscar todas as rodadas registradas na api
    for rodada in range(1, rodada_atual + 1):

        url = f"https://api.cartola.globo.com/atletas/pontuados/{rodada}"
        data = requests.get(url).json()

        atletas = data.get("atletas", {})
        # só trás as pontuações das rodadas com status de finalizadas
        if not atletas:
            print(f"Rodada {rodada} sem pontuação disponível")
            continue

        path = f"{BASE_PATH}/rodada={rodada}"
        
        # cria pasta (ignora se já existir)
        dbutils.fs.mkdirs(path)

        file_name = f"{path}/data_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

        # salva JSON com dbutils (pra salvar volume)
        dbutils.fs.put(file_name, json.dumps(data), overwrite=True)

        print(f"Rodada {rodada} salva com sucesso!")

    print("Pontuação finalizada!")

run()